# synthetic expense data realistic

Load Dataset

In [1]:
import pandas as pd

df = pd.read_csv("../Datasets/synthetic_expense_data_realistic.csv")
df.head()

,Transaction_ID,Date,Time,Amount,Description,Merchant,Category
0,TX0001,4/3/2024,20:57:00,8.53,Lunch meal,McDonalds,Food & Beverage
1,TX0002,8/11/2024,15:39:00,228.41,Electricity bill,Local Utility,Bills
2,TX0003,10/25/2024,18:49:00,109.65,Grocery store purchase,Target,Groceries
3,TX0004,4/4/2024,15:06:00,23.62,Taxi service,Uber,Transportation
4,TX0005,3/26/2025,22:38:00,24.74,Taxi service,Local Transit,Transportation


Remove Duplicates

In [2]:
df.duplicated().sum()

np.int64(0)

Standardize Schema

In [3]:
# Check actual column names
df.columns

Index(['Transaction_ID', 'Date', 'Time', 'Amount', 'Description', 'Merchant',
       'Category'],
      dtype='object')

In [4]:
df = df.rename(columns={
    "Transaction_ID": "TransactionID",
    "Description": "Transaction Description"
})

Drop Time Column

In [5]:
# Time is irrelevant for this research (not doing fraud detection)
df = df.drop(columns=["Time"])

Validate & Fix Date Column

In [6]:
df["Date"].head()

0      4/3/2024
1     8/11/2024
2    10/25/2024
3      4/4/2024
4     3/26/2025
Name: Date, dtype: object

In [7]:
# You must convert to datetime because:
# Behavior analysis uses monthly grouping
# Salary detection uses date gaps
# ML model needs numeric timestamps

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date"])

Validate Amount Column

In [8]:
df["Amount"].describe()

count    1000.000000
mean      125.787180
std       148.796386
min         5.200000
25%        28.502500
50%        70.850000
75%       174.357500
max      1367.850000
Name: Amount, dtype: float64

In [9]:
# Justification , Ensure amount is numeric
# Remove zero or negative values
# Needed for behavior clustering

df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
df = df[df["Amount"] > 0]

Inspect Category Values

In [10]:
df["Category"].unique()

array(['Food & Beverage', 'Bills', 'Groceries', 'Transportation',
       'Healthcare', 'Entertainment'], dtype=object)

In [11]:
# There is no Income-related category.
# This dataset contains ONLY EXPENSE transactions.
df["Type"] = "Expense"

Currency Decision

In [12]:
df["Merchant"].unique()[:20]

array(['McDonalds', 'Local Utility', 'Target', 'Uber', 'Local Transit',
       'Local Clinic', 'Walmart', 'Costco', 'Spotify', 'Dunkin',
       'Netflix', 'Lyft', 'CVS Pharmacy', 'Subway', 'Starbucks',
       'Walgreens', 'AMC Theatres', 'Mobile Carrier', 'Internet Provider'],
      dtype=object)

In [13]:
# These are all United States retail brands → so currency must be USD.
df["Currency"] = "USD"

Add UserID

In [14]:
# This dataset is clearly 1 user (no User column).
# To merge datasets, each dataset must have a USER ID.
df["UserID"] = "US008"

Clean Category Format

In [15]:
df["Category"] = df["Category"].str.title()

Generate TransactionID

In [16]:
df["TransactionID"] = [
    f"SYN{idx:06d}" for idx in range(1, len(df) + 1)
]


Assign Account Name

Categories tell financial behavior:

Groceries → Checking / Credit Card

Transportation → Cash for small amounts

Food & beverage → Cash/Checking

Bills → Checking

Healthcare → Checking

Large payments → savings

In [17]:
def assign_account(row):
    amount = row["Amount"]
    cat = row["Category"].lower()

    if amount > 400:
        return "Savings Account"

    if cat == "bills":
        return "Checking Account"

    if cat == "groceries":
        return "Checking Account" if amount > 100 else "Credit Card"

    if cat == "entertainment":
        return "Credit Card"

    if cat == "transportation":
        return "Cash Wallet" if amount < 30 else "Checking Account"

    if cat == "food & beverage":
        return "Cash Wallet" if amount < 15 else "Checking Account"

    return "Checking Account"

df["Account Name"] = df.apply(assign_account, axis=1)

Validate Required Columns

In [18]:
required = ["TransactionID","UserID","Date","Category","Amount","Type"]
df[required].isna().sum()

TransactionID    0
UserID           0
Date             0
Category         0
Amount           0
Type             0
dtype: int64

In [19]:
df.head()

,TransactionID,Date,Amount,Transaction Description,Merchant,Category,Type,Currency,UserID,Account Name
0,SYN000001,2024-04-03,8.53,Lunch meal,McDonalds,Food & Beverage,Expense,USD,US008,Cash Wallet
1,SYN000002,2024-08-11,228.41,Electricity bill,Local Utility,Bills,Expense,USD,US008,Checking Account
2,SYN000003,2024-10-25,109.65,Grocery store purchase,Target,Groceries,Expense,USD,US008,Checking Account
3,SYN000004,2024-04-04,23.62,Taxi service,Uber,Transportation,Expense,USD,US008,Cash Wallet
4,SYN000005,2025-03-26,24.74,Taxi service,Local Transit,Transportation,Expense,USD,US008,Cash Wallet


Save Final Output

In [20]:
import os
os.makedirs("Tofinal", exist_ok=True)
df.to_csv("Tofinal/synthetic_expense_data_cleaned.csv", index=False)
print("File saved successfully!")

File saved successfully!
